# Attentional Blink

> **Task name:** `Attentional Blink`

**Track: Attention** | Can the model detect a second target shortly after detecting a first?

After identifying a first target (T1) in a rapid sequence, humans experience a
~200-500ms "blink" where a second target (T2) is missed. This benchmark tests whether
LLMs show analogous temporal recovery limitations when processing sequential targets
embedded in distractor streams.

**Metrics:**
- T2|T1 accuracy curve across lags 1-8
- Blink magnitude (max accuracy drop)
- Blink window (lag range where T2 accuracy < baseline)
- Lag-1 sparing (paradoxical preservation at lag 1)

In [ ]:
# Last updated: 2026-03-31 04:59 UTC
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random
import hashlib

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()

def parse_answers(response: str, keys: list[str]) -> dict[str, str]:
    """Parse key: value answers from model response."""
    response = strip_thinking(response)
    result = {}
    for key in keys:
        pattern = rf"{re.escape(key)}\s*[:=]\s*(.+)"
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            result[key] = match.group(1).strip().strip('"').strip("'")
    return result

In [ ]:
# --- Data Generation ---
# Each item is a stream of sentences. T1 and T2 are "target" sentences
# (e.g., about a specific category), distractors are filler sentences.
# Lag = number of distractor items between T1 and T2.

CATEGORIES = {
    "animals": ["The golden eagle soars above mountain peaks at dawn",
                "A Bengal tiger prowls silently through dense bamboo",
                "The blue whale surfaces with a thunderous exhale",
                "A snow leopard traverses the frozen Himalayan ridge",
                "The emperor penguin huddles against Antarctic winds"],
    "cities":  ["Tokyo's neon skyline pulses with midnight energy",
                "Venice's canals reflect centuries of marble architecture",
                "Marrakech's souks overflow with saffron and cedar",
                "Reykjavik glows under the shimmer of northern lights",
                "Buenos Aires echoes with the rhythm of tango music"],
    "foods":   ["The sourdough bread rises slowly in a warm clay oven",
                "Fresh wasabi is grated on sharkskin just before serving",
                "Saffron threads dissolve into the simmering paella rice",
                "The high-altitude chocolate melts at a lower temperature",
                "Aged balsamic vinegar thickens over twelve slow years"],
}

DISTRACTORS = [
    "The quarterly revenue report exceeded analyst expectations by fourteen percent",
    "Parallel processing architectures reduce latency in distributed computing systems",
    "The Renaissance period transformed European artistic and scientific traditions",
    "Tectonic plate movements cause approximately fifty thousand earthquakes annually",
    "The Fibonacci sequence appears in sunflower seed spirals and pinecone geometry",
    "Quantum entanglement enables instantaneous correlation across vast distances",
    "The Industrial Revolution shifted labor from agrarian to manufacturing economies",
    "Photosynthesis converts carbon dioxide and water into glucose using sunlight",
    "The printing press democratized knowledge distribution across social classes",
    "Coral reef ecosystems support approximately twenty-five percent of marine species",
    "Superconductors lose all electrical resistance below critical temperature thresholds",
    "The Silk Road facilitated cultural exchange spanning over four thousand miles",
    "Mitochondrial DNA is inherited exclusively through the maternal lineage",
    "Gothic architecture introduced flying buttresses to support taller cathedral walls",
    "The Doppler effect shifts perceived frequency as a source moves relative to observer",
    "Volcanic eruptions release sulfur dioxide which can temporarily cool global temperatures",
    "The Rosetta Stone provided the key to deciphering Egyptian hieroglyphics in 1822",
    "Enzyme catalysis accelerates biochemical reactions by factors of ten to the seventeen",
    "The Treaty of Westphalia established principles of modern state sovereignty in 1648",
    "Gravitational lensing bends light from distant galaxies around massive foreground objects",
    "The human genome contains approximately three billion base pairs of DNA",
    "Baroque music featured ornate counterpoint and basso continuo accompaniment",
    "Ocean thermohaline circulation distributes heat across hemispheres via deep currents",
    "The discovery of penicillin in 1928 revolutionized treatment of bacterial infections",
]

random.seed(42)
data = []
task_id = 0
LAGS = [1, 2, 3, 4, 5, 6, 7, 8]
STREAM_LENGTH = 20  # total items in each stream

for t1_cat, t2_cat in [("animals", "cities"), ("cities", "foods"), ("foods", "animals")]:
    for t1_idx in range(5):
        for lag in LAGS:
            # Place T1 at position 4-6 (randomized), T2 at T1 + lag + 1
            t1_pos = random.randint(4, 6)
            t2_pos = t1_pos + lag + 1  # +1 because lag=1 means 1 item between

            if t2_pos >= STREAM_LENGTH - 1:
                continue  # skip if T2 would be too close to end

            t1_sentence = CATEGORIES[t1_cat][t1_idx]
            t2_sentence = CATEGORIES[t2_cat][random.randint(0, 4)]

            # Build stream
            stream = []
            used_distractors = random.sample(DISTRACTORS, min(STREAM_LENGTH, len(DISTRACTORS)))
            d_idx = 0
            for pos in range(STREAM_LENGTH):
                if pos == t1_pos:
                    stream.append(t1_sentence)
                elif pos == t2_pos:
                    stream.append(t2_sentence)
                else:
                    stream.append(used_distractors[d_idx % len(used_distractors)])
                    d_idx += 1

            prompt_lines = [f"Item {i+1}: {s}" for i, s in enumerate(stream)]
            prompt = (
                "Below is a rapid stream of sentences. Two of them belong to specific "
                f"categories: one is about **{t1_cat}** (Target 1) and one is about "
                f"**{t2_cat}** (Target 2). All other sentences are unrelated distractors.\n\n"
                + "\n".join(prompt_lines) +
                "\n\nIdentify both targets. Respond in EXACTLY this format:\n"
                f"T1: <the {t1_cat} sentence, verbatim>\n"
                f"T2: <the {t2_cat} sentence, verbatim>"
            )

            data.append({
                "task_id": task_id,
                "prompt": prompt,
                "expected_t1": t1_sentence,
                "expected_t2": t2_sentence,
                "lag": lag,
                "t1_category": t1_cat,
                "t2_category": t2_cat,
                "t1_position": t1_pos,
                "t2_position": t2_pos,
            })
            task_id += 1

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} items across {len(LAGS)} lags")
print(f"Distribution by lag:\n{df_all['lag'].value_counts().sort_index()}")

## Task Definition

In [ ]:
@kbench.task(
    name="attentional_blink",
    description="Dual-target detection in a rapid sequence — measures temporal recovery after first target identification"
)
def attentional_blink(
    llm,
    prompt: str,
    expected_t1: str,
    expected_t2: str,
    lag: int,
) -> tuple[int, int]:
    response = llm.prompt(prompt)
    parsed = parse_answers(response, ["T1", "T2"])

    correct = 0
    total = 2

    # T1 check
    t1_resp = parsed.get("T1", "")
    if expected_t1.lower() in t1_resp.lower() or t1_resp.lower() in expected_t1.lower():
        correct += 1

    # T2 check (only meaningful if T1 was correct — but we score both)
    t2_resp = parsed.get("T2", "")
    if expected_t2.lower() in t2_resp.lower() or t2_resp.lower() in expected_t2.lower():
        correct += 1

    return (correct, total)

## Run Evaluation

In [ ]:
eval_df = df_all[["prompt", "expected_t1", "expected_t2", "lag"]].copy()

print(f"Running {len(eval_df)} items with kbench.llm...")
runs = attentional_blink.evaluate(
    llm=[kbench.llm],
    evaluation_data=eval_df,
    n_jobs=2,
    timeout=120,
    max_attempts=2,
)
results = runs.as_dataframe()
results["accuracy"] = results["result"].apply(
    lambda r: r[0] / r[1] if isinstance(r, tuple) and r[1] > 0 else float(r) if not isinstance(r, tuple) else 0
)
results = results.merge(df_all[["prompt", "lag", "t1_category", "t2_category", "t1_position", "t2_position"]], on="prompt", how="left")
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
print("=== Accuracy by Lag ===")
lag_curve = results.groupby("lag")["accuracy"].agg(["mean", "std", "count"])
print(lag_curve.to_string())

baseline = lag_curve["mean"].max()
blink_window = lag_curve[lag_curve["mean"] < baseline - 0.1].index.tolist()
blink_magnitude = baseline - lag_curve["mean"].min()
lag1_acc = lag_curve.loc[1, "mean"] if 1 in lag_curve.index else None

print(f"\nBaseline accuracy: {baseline:.2%}")
print(f"Blink window (>10% drop): lags {blink_window}")
print(f"Blink magnitude: {blink_magnitude:.2%}")
if lag1_acc is not None:
    print(f"Lag-1 sparing: {lag1_acc:.2%} (paradoxical preservation: {'YES' if lag1_acc > baseline - 0.05 else 'NO'})")

print("\n=== Accuracy by Category Pair ===")
print(results.groupby(["t1_category", "t2_category"])["accuracy"].mean().to_string())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
lag_curve = results.groupby("lag")["accuracy"].mean()
ax.plot(lag_curve.index, lag_curve.values, "o-", linewidth=2, markersize=8, color="steelblue")
ax.axhline(y=lag_curve.max(), color="gray", linestyle="--", alpha=0.5, label=f"Baseline ({lag_curve.max():.0%})")
ax.fill_between(lag_curve.index, lag_curve.values, lag_curve.max(), alpha=0.15, color="red", label="Blink region")
ax.set_xlabel("Lag (items between T1 and T2)")
ax.set_ylabel("T2|T1 Accuracy")
ax.set_title("Attentional Blink Curve")
ax.set_xticks(sorted(results["lag"].unique()))
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig("attentional_blink_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to attentional_blink_curve.png")